# 📓 Notebook 3｜音訊特徵：切幀・頻譜・MFCC・時域（課本 7.5）

> 對應講義 **Part 5**（知識地圖站 5–6）
>
> 全部用合成聲音，不需要任何音檔。目標：做出課本 Fig 7.24–7.27 的「特徵軌跡圖」。

## Step 1｜合成兩段「個性相反」的聲音

- `piano`：穩定和絃（課本 Fig 7.26 的鋼琴 → 特徵軌跡平穩）
- `clap`：拍手聲（課本 Fig 7.24 → 特徵軌跡像雜訊）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import dct

fs = 8000                      # 取樣率 8 kHz（課本語音範例 10 kHz，這裡降一點比較快）
T  = 2.0
t  = np.arange(int(fs * T)) / fs

# 鋼琴感：兩個音的和（220 Hz + 330 Hz + 2 次泛音），振幅慢慢衰減
piano = (np.sin(2*np.pi*220*t) + 0.5*np.sin(2*np.pi*330*t)
         + 0.3*np.sin(2*np.pi*440*t)) * np.exp(-t/3)

# 拍手感：0.35 秒一次的短脈衝（零均值雜訊 → 正負號狂跳 → ZCR 高）
clap = np.zeros_like(piano)
for start in np.arange(0, T, 0.35):
    i = int(start * fs)
    burst = int(0.03 * fs)
    env = np.exp(-np.linspace(0, 6, burst))
    clap[i:i+burst] += env * np.random.default_rng(1).standard_normal(burst)

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
for ax, sig, name in zip(axes, [piano, clap], ['piano 和絃', 'clap 拍手']):
    ax.plot(t, sig, lw=0.6)
    ax.set_title(f'{name}（前 0.5 秒）')
    ax.set_xlim(0, 0.5)
plt.tight_layout()
plt.show()

## Step 2｜切幀 + 窗函數（課本 7.5.1）

把長訊號切成 N=256 點的幀（hop=128，50% 重疊），乘上 Hamming 窗。

In [ ]:
N, hop = 256, 128
def framing(x, N, hop):
    return np.stack([x[i:i+N] for i in range(0, len(x) - N, hop)])

hamm = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(N) / (N - 1))   # 課本 Hamming
# 注意順序：先切幀、再把窗乘到「每一幀」上（整段訊號直接乘窗是常見錯誤）
frames_p = framing(piano, N, hop) * hamm
frames_c = framing(clap, N, hop) * hamm
print('幀數:', frames_p.shape[0], ' 每幀:', frames_p.shape[1], '點')
print(f'時間解析度: 每幀 {N/fs*1000:.1f} ms，幀距 {hop/fs*1000:.1f} ms')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
for ax, fr, name in zip(axes, [frames_p, frames_c], ['piano', 'clap']):
    ax.imshow(fr.T, aspect='auto', origin='lower', cmap='magma',
              extent=[0, fr.shape[0]*hop/fs, 0, N/fs*1000])
    ax.set_ylabel(name + ' 幀 (時間→)')
plt.xlabel('秒'); plt.tight_layout(); plt.show()

## Step 3｜頻譜特徵：Centroid・Roll-off・Flux（課本 7.5.4）

每幀算一個數字，全部手寫（numpy 30 行內搞定）。

In [ ]:
from numpy.fft import rfft, rfftfreq

def spectral_features(frame, fs):
    X = rfft(frame)
    mag = np.abs(X)
    freqs = rfftfreq(len(frame), 1/fs)
    if mag.sum() == 0:                               # 無聲幀 → 特徵定義為 0
        return 0.0, 0.0
    centroid = (freqs * mag).sum() / mag.sum()            # 頻譜重心
    cum = np.cumsum(mag)
    rolloff = freqs[np.searchsorted(cum, 0.85 * cum[-1])] # 85% 能量點
    return centroid, rolloff

def spectral_flux(fr1, fr2):
    n1 = np.abs(rfft(fr1)); n2 = np.abs(rfft(fr2))
    n1 /= n1.max() + 1e-12; n2 /= n2.max() + 1e-12        # 正規化
    return ((n2 - n1) ** 2).sum()                         # 課本 F(i)

cents_p, rols_p = zip(*[spectral_features(f, fs) for f in frames_p])
cents_c, rols_c = zip(*[spectral_features(f, fs) for f in frames_c])
flux_p = [spectral_flux(a, b) for a, b in zip(frames_p[:-1], frames_p[1:])]
flux_c = [spectral_flux(a, b) for a, b in zip(frames_c[:-1], frames_c[1:])]

print(f'piano: centroid 平均 {np.mean(cents_p):7.1f} Hz | rolloff 平均 {np.mean(rols_p):7.1f} Hz')
print(f'clap : centroid 平均 {np.mean(cents_c):7.1f} Hz | rolloff 平均 {np.mean(rols_c):7.1f} Hz')
print('→ clap（寬頻雜訊）的重心遠高於 piano（低頻樂音）✓')

## Step 4｜時域特徵：ZCR・能量（課本 7.5.5）

就是課本 Fig 7.24–7.27 的主角。畫出**特徵隨時間的軌跡**——piano 平穩、clap 躁動。

In [ ]:
def zcr(x):  return np.abs(np.diff(np.sign(x))).sum() / (2 * len(x))
def energy(x): return (x ** 2).mean()

zcr_p, en_p = zip(*[(zcr(f), energy(f)) for f in frames_p])
zcr_c, en_c = zip(*[(zcr(f), energy(f)) for f in frames_c])

fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
tm = np.arange(len(zcr_p)) * hop / fs
for ax, data, name in zip(axes[0], [zcr_p, zcr_c], ['piano ZCR', 'clap ZCR']):
    ax.plot(tm, data); ax.set_title(name); ax.set_ylabel('ZCR')
for ax, data, name in zip(axes[1], [en_p, en_c], ['piano 能量', 'clap 能量']):
    ax.plot(tm, data); ax.set_title(name); ax.set_ylabel('能量')
axes[0][0].set_ylim(0, 0.3)
plt.tight_layout(); plt.show()

### ✏️ 練習：當 30 秒工程師
看上面四張圖——若要寫一個「piano vs clap」分類器，你會不會用 **ZCR 的變異數**（軌跡抖動程度）當特徵？試著算出來：

In [ ]:
print('ZCR 軌跡的變異數：piano =', np.var(zcr_p).round(6), ' clap =', np.var(zcr_c).round(6))
print('→ clap 的 ZCR 軌跡「抖」得多 → 這數字就是一刀切的特徵 ✓')

## Step 5｜音高估計：自相關（課本 7.5.4 的基頻・Brow 91 的窮人版）

樂音有週期性 → 自相關函數在「週期」處有峰值 → 基頻 = fs / 峰值 lag。

In [ ]:
def autocorr_pitch(frame, fs, fmin=60, fmax=500):
    r = np.correlate(frame, frame, 'full')[len(frame)-1:]
    lo, hi = int(fs/fmax), int(fs/fmin)          # 只搜合理音高範圍
    lag = lo + np.argmax(r[lo:hi])
    return fs / lag

# 用「純 220 Hz 單音」測音高估計（混合音會讓自相關出現子諧波，實務上要小心！）
tone = np.sin(2 * np.pi * 220 * t) * np.exp(-t / 3)
frames_tone = framing(tone, N, hop) * hamm
f0 = autocorr_pitch(frames_tone[30], fs)
print(f'純 220 Hz 音的估計基頻 = {f0:.1f} Hz（期望 ≈ 220 ✓）')

## Step 6｜MFCC：手刻梅爾濾波器組（課本 7.5.2–7.5.3）

核心四步：① 幀 → ② mel 三角濾波器組加權 log|X| → ③ DCT → ④ 取前 13 個係數。

In [ ]:
def mel(f):  return 2595 * np.log10(1 + f / 700)          # 課本 (7.65)
def mel_inv(m): return 700 * (10 ** (m / 2595) - 1)

def mel_filterbank(nfilt, nfft, fs):
    mel_pts = np.linspace(mel(0), mel(fs/2), nfilt + 2)      # mel 上等距
    hz = mel_inv(mel_pts)
    bins = np.floor((nfft + 1) * hz / fs).astype(int)        # 對到 FFT bin
    fb = np.zeros((nfilt, nfft // 2 + 1))
    for i in range(1, nfilt + 1):
        for j in range(bins[i-1], bins[i]):
            fb[i-1, j] = (j - bins[i-1]) / (bins[i] - bins[i-1])
        for j in range(bins[i], bins[i+1]):
            fb[i-1, j] = (bins[i+1] - j) / (bins[i+1] - bins[i])
    return fb

fb = mel_filterbank(24, 512, fs)
plt.figure(figsize=(12, 3.5))
plt.imshow(fb, aspect='auto', origin='lower', cmap='viridis')
plt.xlabel('FFT bin（頻率 →）'); plt.ylabel('濾波器編號')
plt.title('24 個 mel 三角濾波器（低頻密、高頻疏）— 講義互動 Demo 8 的靜態版')
plt.colorbar(shrink=0.7); plt.show()

In [ ]:
def mfcc(frame, nfilt=24, ncoef=13):
    X = np.abs(rfft(frame, 512))
    logE = np.log(fb @ X + 1e-10)                  # log(mel 能量) → 課本 (7.67)
    return dct(logE, type=2, norm='ortho')[:ncoef] # DCT = 簡化的逆 DFT → (7.69)

m_p = mfcc(frames_p[30]); m_c = mfcc(frames_c[22])   # 拍手聲挑「脈衝內」的幀（0.35s 處）
print('piano 第 30 幀 MFCC:', np.round(m_p, 3))
print('clap  第 30 幀 MFCC:', np.round(m_c, 3))
print('前 13 個係數就是特徵向量 → 可以丟給分類器了！')

### ⭐ 選修：裝了 librosa 就一行對答案
```bash
pip install librosa
```
（沒裝也能跑完整本筆記本——上面的手刻版就是課本演算法的忠實翻譯。）

In [ ]:
try:
    import librosa
    mfcc_lib = librosa.feature.mfcc(y=piano, sr=fs, n_mfcc=13, n_fft=512, hop_length=hop, n_mels=24)
    print('librosa MFCC（第 30 幀）:', np.round(mfcc_lib[:, 30], 3))
    print('與手刻版的形狀有點差是正常的：濾波器細節/歸一化不同，但「樣子」一致。')
except ImportError:
    print('未安裝 librosa —— 跳過對照（不影響本筆記本）')

## 🏆 本筆記本小結：你現在已經會……
| 技能 | 課本 | 業界 |
|---|---|---|
| 切幀 + Hamming 窗 | 7.5.1 | `librosa.stft` / `scipy.signal.stft` |
| Spectral Centroid / Roll-off / Flux | 7.5.4 | `librosa.feature.spectral_*` |
| ZCR / Energy + 軌跡分析 | 7.5.5 | VAD 語音活動偵測的基本原料 |
| 自相關音高估計 | 7.5.4 | `librosa.pyin` |
| 手刻 MFCC | 7.5.2–7.5.3 | `librosa.feature.mfcc`（業界標準特徵） |

> 下一站：迷你專案 B（拍手 vs 鋼琴判別器）——就把 Step 4 + Step 6 的數字組起來而已！